In [4]:
import os
import numpy as np
import nibabel as nib
from scipy.spatial.distance import pdist
from tqdm import tqdm

from t_testing.clean_roi_mask import modify_mask_with_ttest
import nibabel as nib
import pandas as pd
import logging

from utils.config import Configuration, load_config
from utils.utils import retrieve_stacked_betas
from previousCode.nsddatapaper_rsa.utils.utils import mds

from scipy.stats import wasserstein_distance

# Define custom metric wrapper
def emd(u, v):
    return wasserstein_distance(u, v)

def create_rdm_with_all_trials(
    config: Configuration,
    subj_list,
    mask_value,
    set_to_take,
    t_test_threshold,
    randomization=False
):
    """
    For each subject, load all 3 trials per image (so 150 images ⇒ 450 patterns),
    apply your ROI mask, compute the pairwise RDM (450×450) and MDS coords (450×2).

    Returns
    -------
    results : dict
        { sub : {
            'rdm'      : 1D array of length 450*449/2,
            'mds'      : array (450,2),
            'meta'     : list of (image_id, trial_idx) for each of the 450 rows
          }
        }
    """
    results = {}

    for sub in tqdm(subj_list, desc="Subjects"):
        # 1) Make t-test mask
        pick_shared = (set_to_take == "shared")
        modify_mask_with_ttest(
            config, t_test_threshold, pick_shared, sub, sub_filename="temporary_mask"
        )

        # 2) Load and concatenate LH & RH mask arrays
        roi_dir = config.directories.t_test_roi_dir
        mask_lh = nib.load(os.path.join(roi_dir, set_to_take, f"lh.subj{sub:02d}.temporary_mask.mgz")).get_fdata().squeeze()
        mask_rh = nib.load(os.path.join(roi_dir, set_to_take, f"rh.subj{sub:02d}.temporary_mask.mgz")).get_fdata().squeeze()
        combined_mask = np.concatenate((mask_lh, mask_rh)).astype(int)
        # build a boolean mask of the voxels we want
        vox_selector = (combined_mask == mask_value) if isinstance(mask_value, int) else np.isin(combined_mask, mask_value)
        vox_selector = vox_selector.flatten()

        # 3) Retrieve and stack all three trials
        all_patterns = []
        meta = []   # to store (image_id, trial_idx)
        for trial_idx in range(3):
            betas, image_ids, _ = retrieve_stacked_betas(
                config,
                sub,
                "single",           # single ⇒ one trial at a time
                trial_idx,
                subj_to_check=set_to_take,
                only_face_set=config.pipeline.step_4_rsa_analysis.only_face_set,
                randomization=randomization
            )
            # betas is shape (n_images, V); we want (V, n_images) then transpose to (n_images, V)
            betas = betas.T  # now shape V × 150
            betas = betas.T  # shape (150, V)
            all_patterns.append(betas)
            # replicate image_ids for this trial
            meta += [(img, trial_idx) for img in image_ids]

        # stack into (450, V)
        X450 = np.vstack(all_patterns)  

        # 4) apply the voxel mask and drop any NaNs
        X450_masked = X450[:, vox_selector]
        good_rows = ~np.isnan(X450_masked).any(axis=1)
        X450_masked = X450_masked[good_rows, :]
        meta = [m for keep, m in zip(good_rows, meta) if keep]

        # 5) compute RDM and MDS
        dm = config.pipeline.step_4_rsa_analysis.distance_metric
        metric_fn = emd if dm == "wasserstein" else dm
        rdm_vec = pdist(X450_masked, metric=metric_fn)
        mds_coords = mds(rdm_vec).astype(np.float32)

        # 6) store
        results[sub] = {
            'rdm':    rdm_vec,
            'mds':    mds_coords,
            'meta':   meta
        }

    return results


In [5]:
config = load_config("config.yaml")

results = create_rdm_with_all_trials(
    config,
    subj_list=[1],
    mask_value=7,
    set_to_take="subj_01",
    t_test_threshold=2.5,
    randomization=False
)


Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

2025-05-20 16:19:48 INFO: Processing subject 1 with threshold 2.5
Subjects:   0%|          | 0/1 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'data/t_test_results/subj_01/result_subj_01.npy'